## Read data from Silver and write to Gold

In [0]:
from pyspark.sql import functions as F, Window
from pyspark.sql.functions import struct
from delta.tables import DeltaTable

SILVER_METRICS_TABLE = "tabular.dataexpert.smc_stocks_silver_metrics"
SILVER_PAIRS_TABLE   = "tabular.dataexpert.smc_stocks_silver_pairs"

GOLD_TABLE = "tabular.dataexpert.smc_stocks_gold"
GOLD_PAIR_SUMMARY = "tabular.dataexpert.smc_stocks_gold_pair_summary"

metrics_df = spark.table(SILVER_METRICS_TABLE)
pairs_df = spark.table(SILVER_PAIRS_TABLE)

## Pair Context
The primary gold table will be one row per ticker per day with everything in one place — the OHLCV, the RSI, the volatility, the signal, and enriched pair context like "how many other tickers was this one strongly correlated with today" and "what was its strongest correlation partner."

In [0]:
# --- Pair context: per ticker per day ---
# How many strong partners did this ticker have? Who was its best partner?

#Capturing ticker_a perspective
pair_context_a = (
    pairs_df
    .filter(F.col("intraday_corr") > 0.7)
    .groupBy(F.col("ticker_a").alias("ticker"), "trade_date")
    .agg(
        F.count("*").alias("strong_corr_partner_count"),
        F.max(F.col("intraday_corr")).alias("best_intraday_corr_score"),
        F.max_by("ticker_b", "intraday_corr").alias("best_intraday_corr_partner") # ticker_b that has the highest intraday_corr for each group
    )
)

# F.max_by(col, value_col) returns the value of col corresponding to the maximum of value_col within the group

#Capturing ticker_b perspective
pair_context_b = (
    pairs_df
    .filter(F.col("intraday_corr") > 0.7)
    .groupBy(F.col("ticker_b").alias("ticker"), "trade_date")
    .agg(
        F.count("*").alias("strong_corr_partner_count"),
        F.max(F.col("intraday_corr")).alias("best_intraday_corr_score"),
        F.max_by("ticker_a", "intraday_corr").alias("best_intraday_corr_partner") # ticker_a that has the highest intraday_corr for each group
    )
)

#Combining both
pair_context_full = (
    pair_context_a
    .unionByName(pair_context_b)
    .withColumn(
        "best_pair",
        F.struct("best_intraday_corr_score", "best_intraday_corr_partner")
    )
    .groupBy("ticker", "trade_date")
    .agg(
        F.sum("strong_corr_partner_count").alias("strong_corr_partner_count"),
        F.max_by("best_pair", "best_intraday_corr_score").alias("best_pair")
    )
    .withColumn("best_intraday_corr_score", F.col("best_pair.best_intraday_corr_score"))
    .withColumn("best_intraday_corr_partner", F.col("best_pair.best_intraday_corr_partner"))
    .drop("best_pair")
)

print("Sample data:")
pair_context_full.display(10)

## Creating dataframe for the main gold table (smc_stocks_gold)
- volume_vs_avg_ratio - tells you whether today's volume was unusually high or low for that ticker relative to its own history. Meaning it shows whether the stock was trading normally or with any unusual activity (earnings release, insider trading, breaking news/world events etc.) <br>
- volatility_rank - tells you how volatile a stock is compared to the others for that particular trading day. This ranks stocks by how much they are moving on a given day (high volatility = high risk)

In [0]:

gold_df = (
        metrics_df
        .join(pair_context_full, ["ticker", "trade_date"], "left")
        .withColumn(
            "volume_vs_average_ratio",
            F.round(F.col("day_volume") / F.avg("day_volume").over(Window.partitionBy("ticker")), 4)
        )
        .withColumn(
            "volatility_rank",
            F.percent_rank().over(Window.partitionBy("trade_date").orderBy("volatility_score"))
        )
        .withColumn("gold_ingestion_timestamp", F.current_timestamp())
        .select(
            # Identity
            "ticker", "trade_date",
            # OHLCV
            "day_open", "day_high", "day_low", "day_close",
            "day_volume", "bar_count",
            # Technical indicators
            "rsi", "volatility_score", "volatility_rank",
            # Signal
            "signal_regime", "prev_regime", "regime_changed",
            "prev_rsi", "prev_close", "day_return_pct",
            # Volume context
            "volume_vs_average_ratio",
            # Pair context
            "strong_corr_partner_count", "best_intraday_corr_score", "best_intraday_corr_partner",
            # Metadata
            "gold_ingestion_timestamp",
        )
    )

print("Sample data:")
gold_df.display(10)    

## Creating data frame for the smc_stocks_gold_pair_summary table
- corr_consistency_pct — a pair that shows 80% consistency is a structural relationship, while a pair at 20% is just occasional noise.

In [0]:
gold_pair_summary = (
    pairs_df
    .groupBy("ticker_a", "ticker_b")
    .agg(
        F.count("*").alias("total_days_observed"),
        F.sum(F.when(F.col("intraday_corr") > 0.3, 1).otherwise(0)).alias("strong_corr_days"),
        F.avg("intraday_corr").alias("avg_intraday_corr"),
        F.max(F.col("intraday_corr")).alias("max_intraday_corr"),
        F.avg("return_diff").alias("avg_return_diff"),
        F.min("trade_date").alias("first_observed_date"),
        F.max("trade_date").alias("last_observed_date")
    )
    .withColumn(
        "corr_consistency_pct",
        F.round(F.col("strong_corr_days") / F.col("total_days_observed") * 100, 2)
    )
    .withColumn("gold_ingestion_timestamp", F.current_timestamp())
    .orderBy(F.desc("avg_intraday_corr"))
)

print("Sample data:")
gold_pair_summary.display(10)  

## Write data to gold tables

In [0]:
def merge_to_gold(source_df, target_table, merge_keys:list):
    if spark.catalog.tableExists(target_table):
        delta_table = DeltaTable.forName(spark, target_table)
        merge_condition = " AND ".join([f"target.{key} = source.{key}" for key in merge_keys])
        (
            delta_table.alias("target")
            .merge(
                source_df.alias("source"),
                merge_condition,
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
    else:
        source_df.write.saveAsTable(target_table)
        print(f"Written: {target_table} | Rows: {source_df.count():,}")

merge_to_gold(gold_df, GOLD_TABLE, ["ticker", "trade_date"])
merge_to_gold(gold_pair_summary, GOLD_PAIR_SUMMARY, ["ticker_a", "ticker_b"])

# --- Verify ---
print("\n=== GOLD MAIN ===")
spark.table(GOLD_TABLE).orderBy("trade_date", "ticker").show(10, truncate=False)

print("\n=== GOLD PAIR SUMMARY ===")
spark.table(GOLD_PAIR_SUMMARY).orderBy(F.desc("avg_intraday_corr")).show(10, truncate=False)